# NBA 5-Year RAPM Scraper
**Author:** Dylan Hahami  
**Source:** [nbarapm.com](https://www.nbarapm.com/datasets/six_factor)

Fetches **5-Year 6-Factor Regularized Adjusted Plus-Minus (RAPM)** data from nbarapm.com.

The 6-Factor model breaks each player's impact into:
- Offensive & Defensive efficiency (`sc_OFF_TS`, `sc_DEF_TS`)
- Offensive & Defensive turnovers (`sc_OFF_TOV`, `sc_DEF_TOV`)
- Offensive & Defensive rebounding (`sc_OFF_REB`, `sc_DEF_REB`)

Values represent points per 100 possessions added relative to a league-average player.

### Output
- `data/rapm_5yr.csv` — 5-year RAPM for all qualifying players as of the 2024 endpoint

### Notes
- Filtered to `Latest_Year == 2024` and `Year_Interval == '5Y'` to match the 2019-20 through 2023-24 hustle window.
- Run cells top to bottom.

## 0. Imports & Setup

In [ ]:
import os
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)

print("Libraries loaded.")

## 1. Configuration

The nbarapm.com endpoint returns a JSON array of all player RAPM records across multiple year windows. No authentication is required — only standard browser-like headers.

In [ ]:
URL = "https://www.nbarapm.com/load/SCALEDOUTPUT_SMALLER"

HEADERS = {
    "accept": "*/*",
    "accept-language": "en-US,en;q=0.9",
    "referer": "https://www.nbarapm.com/datasets/six_factor",
    "user-agent": "Mozilla/5.0",
}

# Filter targets — match the 5-year hustle window (2019-20 through 2023-24)
TARGET_YEAR = 2024
TARGET_INTERVAL = "5Y"

## 2. Fetch RAPM Data

In [ ]:
try:
    response = requests.get(URL, headers=HEADERS, timeout=15)
    response.raise_for_status()

    rapm_raw = response.json()

    if not isinstance(rapm_raw, list):
        raise ValueError(f"Expected a JSON list, got {type(rapm_raw).__name__}")

    print(f"Fetched {len(rapm_raw)} total RAPM records.")

except requests.exceptions.HTTPError as e:
    raise SystemExit(f"HTTP error: {e}")
except requests.exceptions.Timeout:
    raise SystemExit("Request timed out — check your connection and retry.")
except Exception as e:
    raise SystemExit(f"Unexpected error: {e}")

## 3. Parse & Filter

The raw data includes multiple year windows (1Y, 3Y, 5Y) and multiple `Latest_Year` snapshots. Filter to the 5-year window ending in 2024 to align with the hustle stats collection period.

In [ ]:
df_raw = pd.DataFrame(rapm_raw)

print(f"Full dataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nAvailable Year_Interval values: {df_raw['Year_Interval'].unique()}")
print(f"Available Latest_Year values:   {sorted(df_raw['Latest_Year'].unique())}")

In [ ]:
rapm_5yr = df_raw[
    (df_raw["Latest_Year"] == TARGET_YEAR) &
    (df_raw["Year_Interval"] == TARGET_INTERVAL)
].copy()

print(f"Filtered to {TARGET_INTERVAL} / Latest_Year={TARGET_YEAR}: {len(rapm_5yr)} players")

if rapm_5yr.empty:
    raise ValueError(
        f"No records found for Year_Interval='{TARGET_INTERVAL}' and Latest_Year={TARGET_YEAR}.\n"
        "Check the available values printed above and update the filter constants if needed."
    )

rapm_5yr.head()

## 4. Preview Key Columns

Confirm the 6-Factor components and overall RAPM are present before saving.

In [ ]:
KEY_COLS = [
    "sc_OFF_TS", "sc_DEF_TS",
    "sc_OFF_TOV", "sc_DEF_TOV",
    "sc_OFF_REB", "sc_DEF_REB",
    "OVR_RAPM",
]

missing = [c for c in KEY_COLS if c not in rapm_5yr.columns]
if missing:
    print(f"Warning: expected columns not found: {missing}")
    print(f"Available columns: {list(rapm_5yr.columns)}")
else:
    print("All expected 6-Factor columns present.")
    display(rapm_5yr[KEY_COLS].describe().round(3))

## 5. Save to CSV

In [ ]:
output_path = "data/rapm_5yr.csv"
rapm_5yr.to_csv(output_path, index=False)
print(f"Saved {len(rapm_5yr)} rows to '{output_path}'")